# PSA Translation — mT5-small Fine-Tuning
### English/Kiswahili → Ekegusii (low-resource) machine translation

This notebook trains and evaluates **mT5-small** with layer freezing on a curated
English/Kiswahili → Ekegusii PSA (Public Service Announcement) dataset.

**No Colab, Kaggle, or Weights & Biases dependency** — designed to run on any standard
Python + GPU environment (e.g. Navon Cloud JupyterLab). All logs, metrics, and checkpoints
are written to local files under `./checkpoints/` and `./logs/`.

**Requirements:** Python 3.10+, a CUDA GPU (tested on NVIDIA T4 15GB; will run faster/larger
batches on an A100), and `Final_merged_psas.csv` placed in the same directory as this notebook
(or update `DATA_PATH` below).


## 1. Setup

In [1]:
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate mlflow scikit-learn


In [2]:
import os
# Restrict to a single GPU by default -- safe no-op on single-GPU machines,
# and avoids a known multi-GPU memory-overhead issue on some setups.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


'0'

In [3]:
import os, json, time, random
import numpy as np
import pandas as pd
import torch

from transformers import set_seed

# Reproducibility
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Local output directories (created automatically, no cloud mounts needed)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: True
Device: NVIDIA A100-SXM4-80GB


## 2. Load curated dataset

In [4]:
# Place Final_merged_psas.csv in the same folder as this notebook,
# or set DATA_PATH to its full path.
DATA_PATH = "Final_merged_psas.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()


(21306, 5)


,PSA_ID,Domain,English,Kiswahili,Ekegusii
0,1,Agriculture,Farmers are urged to prioritize safe agrochemi...,Wakulima wanakumbushwa kuzingatia matumizi sal...,Abakuli batosere omogori bw'ogenda gesia bw'og...
1,2,Agriculture,Trucks ferrying top-dressing fertilizer are no...,Masafa yanayosafirisha mbolea ya kuongeza mavu...,Matika agwanana oria okora buya bwakonyang'ana...
2,3,Agriculture,Farmers in Wajir are invited to participate in...,Wakulima wa Wajir wanakaribishwa kushiriki kat...,Abagere bu Wajir batarikire kugana mu Ksh. 5 b...
3,4,Agriculture,Farmers are encouraged to participate in the s...,Wakulima wanahimizwa kushiriki katika mpango w...,Abagaba batemerewe kuhakanya mulashi wa ethano...
4,5,Agriculture,A Ksh. 34.4 billion program has been launched ...,Mpango wa Ksh. bilioni 34.4 umeanzishwa ili ku...,Programu ya Ksh. 34.4 bilioni imeanzishwa kuim...


### 2.1 Build the combined (English + Kiswahili) → Ekegusii dataset

Each row becomes **two** training examples where possible: one with English as source,
one with Kiswahili as source, both mapping to the same Ekegusii target.

In [7]:
def build_combined(df):
    rows = []
    for _, r in df.iterrows():
        if pd.notna(r["English"]) and str(r["English"]).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["English"], "target_text": r["Ekegusii"],
                "source_lang": "en"
            })
        if pd.notna(r.get("Kiswahili")) and str(r.get("Kiswahili")).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["Kiswahili"], "target_text": r["Ekegusii"],
                "source_lang": "sw"
            })
    return pd.DataFrame(rows).dropna(subset=["target_text"])

combined = build_combined(df)
combined = combined[combined["target_text"].astype(str).str.strip() != ""]
print("Total combined examples:", len(combined))
print(combined["source_lang"].value_counts())
print(combined["Domain"].value_counts())


Total combined examples: 42609
source_lang
en    21306
sw    21303
Name: count, dtype: int64
Domain
Education            10573
Agriculture           8912
Health                8176
Security & Safety     8028
Governance            6920
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import GroupShuffleSplit

# Split by PSA_ID (not by row) so the English and Kiswahili versions of the same
# PSA -- which share the identical Ekegusii target -- always land in the same split.
# Splitting on rows directly would leak target sentences across train/test.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss.split(combined, groups=combined["PSA_ID"]))
train_df, temp_df = combined.iloc[train_idx].reset_index(drop=True), combined.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["PSA_ID"]))
val_df, test_df = temp_df.iloc[val_idx].reset_index(drop=True), temp_df.iloc[test_idx].reset_index(drop=True)

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
print(train_df["source_lang"].value_counts())
print(test_df["source_lang"].value_counts())

# Sanity check: confirm no PSA_ID appears in more than one split
assert set(train_df["PSA_ID"]) & set(val_df["PSA_ID"]) == set()
assert set(train_df["PSA_ID"]) & set(test_df["PSA_ID"]) == set()
assert set(val_df["PSA_ID"]) & set(test_df["PSA_ID"]) == set()
print("No PSA_ID overlap between splits -- confirmed.")


Train: 34081  Val: 4256  Test: 4272
source_lang
en    17042
sw    17039
Name: count, dtype: int64
source_lang
en    2136
sw    2136
Name: count, dtype: int64
No PSA_ID overlap between splits -- confirmed.


**Low-resource note:** Ekegusii is not in mT5's pretraining language list,
so this is a genuine low-resource target. English is well covered by mT5's pretraining;
Kiswahili is present but underrepresented relative to English.

## 3. Shared utilities: tokenization, metrics, layer freezing, timing

In [9]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate

sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def to_hf(d):
    return Dataset.from_pandas(d[["source_text", "target_text", "source_lang", "Domain"]]
                                .reset_index(drop=True))

train_ds = to_hf(train_df)
val_ds   = to_hf(val_df)
test_ds  = to_hf(test_df)

def build_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        # -100 is the label-ignore sentinel; must be swapped for a real pad id before decoding
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        c = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        return {"bleu": bleu["score"], "chrf": c["score"]}
    return compute_metrics

def freeze_encoder_layers(model, num_layers_to_freeze):
    """Freeze bottom N encoder layers to reduce overfitting risk on our small,
    low-resource fine-tuning set and cut compute cost."""
    encoder = model.get_encoder()
    layers = encoder.block if hasattr(encoder, "block") else encoder.layers
    for i, layer in enumerate(layers):
        if i < num_layers_to_freeze:
            for p in layer.parameters():
                p.requires_grad = False
    return model

def score(preds, refs, label):
    """Computes BLEU/chrF, prints, and returns a result dict (no external logging service)."""
    bleu = sacrebleu.compute(predictions=preds, references=[[r] for r in refs])
    c = chrf.compute(predictions=preds, references=[[r] for r in refs])
    print(f"{label:35s} BLEU={bleu['score']:.2f}  chrF={c['score']:.2f}")
    return {"name": label, "bleu": bleu["score"], "chrf": c["score"]}

MAX_LEN = 128
results_log = []          # collects every score() call for the final summary table
timing_log = {}           # collects wall-clock training time


## 4. mT5-small — tokenizer and preprocessing

In [10]:
MT5_CHECKPOINT = "google/mt5-small"
mt5_tok = AutoTokenizer.from_pretrained(MT5_CHECKPOINT)

def mt5_prefix(source_lang):
    return "translate English to Ekegusii: " if source_lang == "en" else "translate Kiswahili to Ekegusii: "

def preprocess_mt5(batch):
    inputs = [mt5_prefix(sl) + t for sl, t in zip(batch["source_lang"], batch["source_text"])]
    model_inputs = mt5_tok(inputs, max_length=MAX_LEN, truncation=True)
    labels = mt5_tok(text_target=batch["target_text"], max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok_mt5 = train_ds.map(preprocess_mt5, batched=True)
val_tok_mt5   = val_ds.map(preprocess_mt5, batched=True)


Map: 100%|██████████| 4256/4256 [00:00<00:00, 13461.78 examples/s]


In [11]:
import mlflow

mlflow.set_experiment("psa-translation-mt5-en-sw-to-guz")
mlflow.start_run(run_name="mt5_combined_guz")
mlflow.log_params({
    "model_checkpoint": MT5_CHECKPOINT,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
})
# Per-epoch training/eval metrics (loss, BLEU, chrF) are logged automatically by the
# HF Trainer's built-in MLflow integration once report_to=["mlflow"] is set below --
# no manual per-step logging needed.


2026/08/02 00:10:56 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/08/02 00:10:56 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

### 4.1 Baseline (zero-shot) — mT5, before any fine-tuning

In [12]:
base_mt5 = AutoModelForSeq2SeqLM.from_pretrained(MT5_CHECKPOINT)
base_mt5.to("cuda" if torch.cuda.is_available() else "cpu")

def generate_mt5(model, texts, source_langs, max_new_tokens=MAX_LEN, batch_size=16):
    inputs = [mt5_prefix(sl) + t for sl, t in zip(source_langs, texts)]
    all_preds = []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i + batch_size]
        enc = mt5_tok(batch, return_tensors="pt", padding=True, truncation=True,
                      max_length=MAX_LEN).to(model.device)
        out = model.generate(**enc, max_length=max_new_tokens)
        all_preds.extend(mt5_tok.batch_decode(out, skip_special_tokens=True))
        del enc, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return all_preds

# Evaluate baseline separately for each source language (needed for domain/ablation tables)
test_en = test_df[test_df["source_lang"] == "en"]
test_sw = test_df[test_df["source_lang"] == "sw"]

preds_base_en = generate_mt5(base_mt5, list(test_en["source_text"]), list(test_en["source_lang"]))
results_log.append(score(preds_base_en, list(test_en["target_text"]), "mt5_zero-shot_en-guz"))

preds_base_sw = generate_mt5(base_mt5, list(test_sw["source_text"]), list(test_sw["source_lang"]))
results_log.append(score(preds_base_sw, list(test_sw["target_text"]), "mt5_zero-shot_sw-guz"))

del base_mt5
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights: 100%|██████████| 192/192 [00:00<00:00, 24314.81it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mt5_zero-shot_en-guz                BLEU=0.00  chrF=1.18
mt5_zero-shot_sw-guz                BLEU=0.01  chrF=1.44


### 4.2 Fine-tuning — mT5 (few-shot), with layer freezing

Checkpoints save automatically each epoch to `checkpoints/mt5_combined_guz/`.
Training logs (loss, BLEU, chrF per epoch) are written to `logs/mt5_training_log.csv`
after training completes — no external logging service required.

In [13]:
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_CHECKPOINT)
mt5_model = freeze_encoder_layers(mt5_model, num_layers_to_freeze=4)  # mt5-small: 8 encoder layers

data_collator_mt5 = DataCollatorForSeq2Seq(mt5_tok, model=mt5_model)

args_mt5 = Seq2SeqTrainingArguments(
    output_dir="checkpoints/mt5_combined_guz",
    per_device_train_batch_size=32,  # bumped from 8 -- A100 80GB has ample headroom for mT5-small
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    num_train_epochs=5,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    report_to=["mlflow"],   # auto-logs per-epoch loss/BLEU/chrF to the active MLflow run            # no external logging service
    fp16=False,                  # mT5 is unstable in fp16 on T4-class GPUs -- train in fp32
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    gradient_checkpointing=False,  # unnecessary on 80GB for a ~300M-param model; was needed only on 15GB T4
    bf16=True,  # A100 supports bf16 natively -- same exponent range as fp32, avoids the fp16 NaN issue
)

trainer_mt5 = Seq2SeqTrainer(
    model=mt5_model,
    args=args_mt5,
    train_dataset=train_tok_mt5,
    eval_dataset=val_tok_mt5,
    data_collator=data_collator_mt5,
    processing_class=mt5_tok,
    compute_metrics=build_compute_metrics(mt5_tok),
)

t0 = time.time()
trainer_mt5.train()
timing_log["mt5_train_seconds"] = time.time() - t0
print(f"mT5 training time: {timing_log['mt5_train_seconds']/60:.1f} minutes")

# Save the best checkpoint (load_best_model_at_end=True already loaded it into memory) as a
# clean, inference-ready model -- this is what the demo function below loads.
trainer_mt5.save_model("checkpoints/mt5_combined_guz/final")
mt5_tok.save_pretrained("checkpoints/mt5_combined_guz/final")

# --- Save training log locally (replaces the W&B dashboard) ---
log_df = pd.DataFrame(trainer_mt5.state.log_history)
log_df.to_csv("logs/mt5_training_log.csv", index=False)
print("Training log saved to logs/mt5_training_log.csv")


Loading weights: 100%|██████████| 192/192 [00:00<00:00, 26900.93it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,4.040248,3.720722,1.712680,21.079013
2,3.732924,3.507565,2.386553,22.578745
3,3.572714,3.414613,2.620530,23.929362
4,3.525028,3.365422,2.985632,24.637274
5,3.495724,3.347350,3.112052,25.333702


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.61s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


mT5 training time: 89.9 minutes


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it]

Training log saved to logs/mt5_training_log.csv


### 4.3 Fine-tuned evaluation — mT5, per source language

In [14]:
preds_ft_en = generate_mt5(trainer_mt5.model, list(test_en["source_text"]), list(test_en["source_lang"]))
results_log.append(score(preds_ft_en, list(test_en["target_text"]), "mt5_few-shot_en-guz"))

preds_ft_sw = generate_mt5(trainer_mt5.model, list(test_sw["source_text"]), list(test_sw["source_lang"]))
results_log.append(score(preds_ft_sw, list(test_sw["target_text"]), "mt5_few-shot_sw-guz"))

# Persist all results (zero-shot + few-shot) locally
with open("logs/mt5_results.json", "w") as f:
    json.dump(results_log, f, indent=2)
print("Results saved to logs/mt5_results.json")


mt5_few-shot_en-guz                 BLEU=3.08  chrF=25.79
mt5_few-shot_sw-guz                 BLEU=2.69  chrF=24.57
Results saved to logs/mt5_results.json


### 4.4 Domain ablation — per-domain performance (fine-tuned model)

Checks whether performance holds up across PSA domains (health, agriculture, etc.),
not just in aggregate — a lightweight stand-in for full domain-adaptation analysis.


In [15]:
print("=== Domain ablation (few-shot mt5, combined en+sw) ===")
domain_results = []
for domain, group in test_df.groupby("Domain"):
    preds = generate_mt5(trainer_mt5.model, list(group["source_text"]), list(group["source_lang"]))
    r = score(preds, list(group["target_text"]), f"mt5_domain_{domain}")
    r["n"] = len(group)
    domain_results.append(r)

domain_df = pd.DataFrame(domain_results)
domain_df.to_csv("logs/mt5_domain_ablation.csv", index=False)
mlflow.log_artifact("logs/mt5_domain_ablation.csv")
domain_df


=== Domain ablation (few-shot mt5, combined en+sw) ===
mt5_domain_Agriculture              BLEU=2.23  chrF=21.90
mt5_domain_Education                BLEU=3.44  chrF=25.42
mt5_domain_Governance               BLEU=4.14  chrF=30.82
mt5_domain_Health                   BLEU=3.28  chrF=25.98
mt5_domain_Security & Safety        BLEU=1.92  chrF=23.91


,name,bleu,chrf,n
0,mt5_domain_Agriculture,2.231921,21.899330,944
1,mt5_domain_Education,3.442041,25.421971,1058
2,mt5_domain_Governance,4.141700,30.822579,660
3,mt5_domain_Health,3.279891,25.984319,768
4,mt5_domain_Security & Safety,1.917826,23.908534,842


### 4.5 Save full test-set predictions + confidence scores (for COMET / human eval / error analysis)

Generates translations for the **entire** test set (not the capped subsets used in the ablation
cells above) with the fine-tuned model, plus a per-sentence confidence score, and writes it all
to a CSV. This file is the raw material for next week's COMET scoring, native-speaker evaluation,
and error analysis -- none of those need the model reloaded or re-run once this exists.


In [16]:
def generate_with_confidence_mt5(model, texts, source_langs, batch_size=16):
    """Like generate_mt5, but also returns a per-sentence confidence score
    (mean exponentiated log-probability of the generated tokens, roughly 0-1)."""
    inputs = [mt5_prefix(sl) + t for sl, t in zip(source_langs, texts)]
    all_preds, all_conf = [], []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i + batch_size]
        enc = mt5_tok(batch, return_tensors="pt", padding=True, truncation=True,
                      max_length=MAX_LEN).to(model.device)
        out = model.generate(**enc, max_length=MAX_LEN,
                              output_scores=True, return_dict_in_generate=True)
        all_preds.extend(mt5_tok.batch_decode(out.sequences, skip_special_tokens=True))
        transition_scores = model.compute_transition_scores(
            out.sequences, out.scores, normalize_logits=True
        )
        # Positions past the end of a shorter sequence in the batch score as ~-inf; mask them out
        mask = transition_scores > -1e9
        seq_logprob = (transition_scores * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        all_conf.extend(torch.exp(seq_logprob).tolist())
        del enc, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return all_preds, all_conf


preds_mt5_full, conf_mt5_full = generate_with_confidence_mt5(
    trainer_mt5.model, list(test_df["source_text"]), list(test_df["source_lang"])
)

predictions_df = test_df[["source_text", "source_lang", "Domain", "target_text"]].copy()
predictions_df = predictions_df.rename(columns={"target_text": "reference"})
predictions_df["mt5_prediction"] = preds_mt5_full
predictions_df["mt5_confidence"] = conf_mt5_full

predictions_df.to_csv("logs/mt5_test_predictions.csv", index=False)
mlflow.log_artifact("logs/mt5_test_predictions.csv")
print(f"Saved {len(predictions_df)} test-set predictions with confidence scores "
      f"to logs/mt5_test_predictions.csv")
predictions_df.head()


Saved 4272 test-set predictions with confidence scores to logs/mt5_test_predictions.csv


,source_text,source_lang,Domain,reference,mt5_prediction,mt5_confidence
0,The 4th National Kalro Open Research Week and ...,en,Agriculture,Naki za Nane Kaliro Open Research Week na Naki...,Ekeombe keria ekenene getenenerete korangeria ...,0.003638
1,Siku ya 4 ya Utafiti wa Taifa wa Kalro na Maon...,sw,Agriculture,Naki za Nane Kaliro Open Research Week na Naki...,Ekitabu ky'oboremi 4 ky'oboremi bw'ekirori na ...,0.449231
2,Kenya is set to enhance cotton production and ...,en,Agriculture,Kenya ikirongo omokebura ibori naki omokoya. A...,Kenya ebuya oria oria oria oria oria oria oria...,0.579613
3,Kenya inaandaa kuboresha uzalishaji wa pamba n...,sw,Agriculture,Kenya ikirongo omokebura ibori naki omokoya. A...,Kenya ebuya okora ebikora na ebikora bya ebime...,0.431686
4,1 million bags of subsidized fertilizers are n...,en,Agriculture,Maki 1 milioni ekeria chichieri kyabere buya b...,Abakire 1 million abaki abaki abaki abaki abak...,0.397870


## 5. Hyperparameters, training time, and results

Auto-generated from what actually ran, and saved to `logs/mt5_hyperparameters.csv`
and `logs/mt5_results_table.csv` for the write-up.

In [17]:
hyperparam_table = pd.DataFrame([{
    "Model": "mT5-small",
    "Pair": "combined (en+sw)->guz",
    "Epochs": args_mt5.num_train_epochs,
    "Batch size": args_mt5.per_device_train_batch_size,
    "Learning rate": args_mt5.learning_rate,
    "Frozen encoder layers": "4/8",
    "fp16": args_mt5.fp16,
    "Gradient checkpointing": args_mt5.gradient_checkpointing,
    "Train time (min)": round(timing_log.get("mt5_train_seconds", 0) / 60, 1),
}])
hyperparam_table.to_csv("logs/mt5_hyperparameters.csv", index=False)
hyperparam_table


,Model,Pair,Epochs,Batch size,Learning rate,Frozen encoder layers,fp16,Gradient checkpointing,Train time (min)
0,mT5-small,combined (en+sw)->guz,5,32,0.0003,4/8,False,False,89.9


In [18]:
results_df = pd.DataFrame(results_log)
results_df.to_csv("logs/mt5_results_table.csv", index=False)
print("=== Zero-shot vs Few-shot results ===")
print(results_df.to_string(index=False))


=== Zero-shot vs Few-shot results ===
                name     bleu      chrf
mt5_zero-shot_en-guz 0.004645  1.176074
mt5_zero-shot_sw-guz 0.013234  1.444270
 mt5_few-shot_en-guz 3.075943 25.794362
 mt5_few-shot_sw-guz 2.687749 24.569205


In [21]:
import re
domain_df["name"] = domain_df["name"].apply(lambda s: re.sub(r"[^A-Za-z0-9_\-. :/]", "_", str(s)))

In [22]:
# --- Final MLflow logging: extra hyperparameters not auto-captured, plus result artifacts ---
mlflow.log_params({
    "frozen_encoder_layers": "4/8",
    "gradient_checkpointing": args_mt5.gradient_checkpointing,
    "bf16": args_mt5.bf16,
    "fp16": args_mt5.fp16,
    "batch_size": args_mt5.per_device_train_batch_size,
})
mlflow.log_metrics({f"{r['name']}_bleu": r['bleu'] for r in results_log})
mlflow.log_metrics({f"{r['name']}_chrf": r['chrf'] for r in results_log})
mlflow.log_metrics({f"{row['name']}_bleu": row['bleu'] for _, row in domain_df.iterrows()})
mlflow.log_metrics({f"{row['name']}_chrf": row['chrf'] for _, row in domain_df.iterrows()})
for fpath in ["logs/mt5_hyperparameters.csv", "logs/mt5_results_table.csv", "logs/mt5_training_log.csv"]:
    mlflow.log_artifact(fpath)

mlflow.end_run()
print("MLflow run closed. Run `mlflow ui` in a terminal in this directory to view it.")


MLflow run closed. Run `mlflow ui` in a terminal in this directory to view it.


## 6. Inference demo

Loads the fine-tuned model from `checkpoints/mt5_combined_guz/final` and translates
sample sentences. Run this cell independently (after training, or in a fresh session
that has skipped straight to this section) to demonstrate translation on new input.

In [20]:
FINAL_MODEL_DIR = "checkpoints/mt5_combined_guz/final"

# If this cell is run standalone (e.g. a fresh kernel after training already happened),
# reload the fine-tuned model + tokenizer from disk instead of relying on in-memory objects.
if "trainer_mt5" not in globals():
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    mt5_tok = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
    _mt5_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL_DIR)
    _mt5_model.to("cuda" if torch.cuda.is_available() else "cpu")
else:
    _mt5_model = trainer_mt5.model

def mt5_prefix(source_lang):
    return "translate English to Ekegusii: " if source_lang == "en" else "translate Kiswahili to Ekegusii: "

def translate_psa(text, source_lang="en"):
    """
    text: input sentence (English or Kiswahili)
    source_lang: 'en' (English) or 'sw' (Kiswahili)
    Returns: Ekegusii translation from the fine-tuned mT5 model
    """
    inputs = mt5_tok(mt5_prefix(source_lang) + text, return_tensors="pt",
                      truncation=True, max_length=MAX_LEN).to(_mt5_model.device)
    out = _mt5_model.generate(**inputs, max_length=MAX_LEN)
    return mt5_tok.decode(out[0], skip_special_tokens=True)

# --- Demo: sample PSAs ---
samples = [
    ("Farmers are urged to prioritize safe agrochemical usage this season.", "en"),
    ("Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.", "sw"),
]

for text, lang in samples:
    print(f"[{lang}] {text}")
    print("  mT5 ->", translate_psa(text, source_lang=lang))
    print()


[en] Farmers are urged to prioritize safe agrochemical usage this season.
  mT5 -> Abagere bakire buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buy

[sw] Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.
  mT5 -> Abagere bakire buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buya buy

